In [1]:
!pip install -q torch \
 transformers \
 bitsandbytes \
 nvidia-ml-py \
 langchain-groq \
 langchain-google-genai \
 langchain-mistralai

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 20.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 70.5/70.5 kB 7.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.5/137.5 kB 7.7 MB/s eta 0:00:00


In [2]:
!pip install -q --upgrade transformers accelerate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.2/11.2 MB 39.5 MB/s eta 0:00:00


In [3]:
import torch
import time
import re
import threading
import pandas as pd
import pynvml
from google.colab import userdata
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

from langchain_groq import ChatGroq
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_mistralai import ChatMistralAI
from langchain.messages import HumanMessage

print("Imports successful")

Imports successful


In [4]:
! nvidia-smi

Mon Jun 29 19:09:55 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   37C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [5]:
df = pd.read_json('benchmark.json')
print(df["prompt"][1])

If a clock strikes 6 times in 5 seconds, how many seconds will it take to strike 12 times?


In [6]:
quantization_config = BitsAndBytesConfig(
    load_in_8bit=True,
)

In [7]:

# ── Load model ───────────────────────────────────────────────────────
tokenizer = AutoTokenizer.from_pretrained("deepseek-ai/DeepSeek-R1-Distill-Llama-8B")
model = AutoModelForCausalLM.from_pretrained(
    "deepseek-ai/DeepSeek-R1-Distill-Llama-8B",
    quantization_config=quantization_config,
    device_map='auto'
)

# ── GPU monitor setup ────────────────────────────────────────────────
pynvml.nvmlInit()
handle = pynvml.nvmlDeviceGetHandleByIndex(0)

# ── Helper: poll GPU in background thread during generation ──────────
def poll_gpu(handle, interval, results, stop_event):
    """Polls GPU every `interval` seconds until stop_event is set."""
    util_readings, temp_readings, mem_readings = [], [], []
    while not stop_event.is_set():
        util_readings.append(pynvml.nvmlDeviceGetUtilizationRates(handle).gpu)
        temp_readings.append(pynvml.nvmlDeviceGetTemperature(handle, pynvml.NVML_TEMPERATURE_GPU))
        mem_readings.append(pynvml.nvmlDeviceGetMemoryInfo(handle).used / 1024**2)
        time.sleep(interval)
    results['gpu_usage_pct']   = round(max(util_readings), 2)   # peak usage
    results['gpu_temp_c']      = round(max(temp_readings), 2)   # peak temp
    results['gpu_mem_used_mb'] = round(max(mem_readings), 2)    # peak memory

# ── Strip DeepSeek <think> chain-of-thought block ───────────────────
def strip_think_block(text):
    return re.sub(r'<think>.*?</think>', '', str(text), flags=re.DOTALL).strip()

# ── Main inference loop ──────────────────────────────────────────────
deepseek_records = []

for i in range(len(df)):
    inputs = tokenizer(
        df["prompt"][i],
        return_tensors='pt'
    ).to(model.device)

    prompt_tokens = inputs["input_ids"].shape[1]

    # Start background GPU polling
    gpu_results = {}
    stop_event  = threading.Event()
    gpu_thread  = threading.Thread(
        target=poll_gpu,
        args=(handle, 0.5, gpu_results, stop_event)  # poll every 0.5s
    )
    gpu_thread.start()

    # Generation
    start = time.time()
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=2048,
            pad_token_id=tokenizer.eos_token_id   # avoids warning on Qwen
        )
    latency = time.time() - start

    # Stop GPU polling
    stop_event.set()
    gpu_thread.join()

    # Decode only new tokens (strips prompt echo)
    new_tokens    = outputs[0][prompt_tokens:]
    output_tokens = new_tokens.shape[0]
    text          = tokenizer.decode(new_tokens, skip_special_tokens=True)
    text          = strip_think_block(text)

    deepseek_records.append({
        "question_id"     : i,
        "question"        : df["prompt"][i],
        "answer"          : text,
        "prompt_tokens"   : prompt_tokens,
        "output_tokens"   : output_tokens,
        "total_tokens"    : prompt_tokens + output_tokens,
        "latency_sec"     : round(latency, 4),
        "tokens_per_sec"  : round(output_tokens / latency, 2),
        "gpu_usage_pct"   : gpu_results.get('gpu_usage_pct'),
        "gpu_temp_c"      : gpu_results.get('gpu_temp_c'),
        "gpu_mem_used_mb" : gpu_results.get('gpu_mem_used_mb'),
    })

    print(f"[{i+1}/{len(df)}] done | "
          f"out_tokens: {output_tokens} | "
          f"latency: {latency:.2f}s | "
          f"tok/s: {output_tokens/latency:.1f} | "
          f"gpu: {gpu_results.get('gpu_usage_pct')}% | "
          f"temp: {gpu_results.get('gpu_temp_c')}C | "
          f"mem: {gpu_results.get('gpu_mem_used_mb')}MB")

deepseek_df = pd.DataFrame(deepseek_records)
deepseek_df['model'] = 'deepseek'
deepseek_df.to_csv('deepseek_metrics.csv', index=False)
print("\nSaved deepseek_metrics.csv")
print(deepseek_df[['question_id','output_tokens','latency_sec',
                    'tokens_per_sec','gpu_usage_pct','gpu_temp_c',
                    'gpu_mem_used_mb']].to_string())

config.json:   0%|          | 0.00/826 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/3.07k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.08M [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/24.2k [00:00<?, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/181 [00:00<?, ?B/s]

/usr/local/lib/python3.12/dist-packages/bitsandbytes/autograd/_functions.py:123: UserWarning: MatMul8bitLt: inputs will be cast from torch.bfloat16 to float16 during quantization
  warnings.warn(f"MatMul8bitLt: inputs will be cast from {A.dtype} to float16 during quantization")


[1/30] done | out_tokens: 2048 | latency: 383.50s | tok/s: 5.3 | gpu: 56% | temp: 79C | mem: 9778.19MB
[2/30] done | out_tokens: 349 | latency: 61.41s | tok/s: 5.7 | gpu: 43% | temp: 78C | mem: 9778.19MB
[3/30] done | out_tokens: 169 | latency: 28.78s | tok/s: 5.9 | gpu: 43% | temp: 79C | mem: 9778.19MB
[4/30] done | out_tokens: 2048 | latency: 382.34s | tok/s: 5.4 | gpu: 51% | temp: 79C | mem: 9778.19MB
[5/30] done | out_tokens: 2048 | latency: 379.38s | tok/s: 5.4 | gpu: 51% | temp: 80C | mem: 9778.19MB
[6/30] done | out_tokens: 2000 | latency: 362.09s | tok/s: 5.5 | gpu: 53% | temp: 81C | mem: 9778.19MB
[7/30] done | out_tokens: 527 | latency: 91.69s | tok/s: 5.7 | gpu: 44% | temp: 81C | mem: 9782.19MB
[8/30] done | out_tokens: 1897 | latency: 336.94s | tok/s: 5.6 | gpu: 52% | temp: 82C | mem: 9782.19MB
[9/30] done | out_tokens: 2048 | latency: 370.48s | tok/s: 5.5 | gpu: 52% | temp: 82C | mem: 9782.19MB
[10/30] done | out_tokens: 2048 | latency: 365.57s | tok/s: 5.6 | gpu: 53% | te

In [8]:
deepseek_df.head()

,question_id,question,answer,prompt_tokens,output_tokens,total_tokens,latency_sec,tokens_per_sec,gpu_usage_pct,gpu_temp_c,gpu_mem_used_mb,model
0,0,A train leaves Station A traveling at 60 mph. ...,"A)Ġ1.5B)Ġ2C)Ġ3D)Ġ4ĊĊOkay,ĠsoĠIĠhaveĠthisĠprobl...",53,2048,2101,383.5002,5.34,56,79,9778.19,deepseek
1,1,"If a clock strikes 6 times in 5 seconds, how m...","**ĊĊToĠsolveĠthis,ĠIĠneedĠtoĠdetermineĠhowĠman...",23,349,372,61.4077,5.68,43,78,9778.19,deepseek
2,2,All prompt engineers are tech enthusiasts. Som...,"Yes,that'scorrect.ĊĊ</think>ĊĊBasedĠonĠtheĠgiv...",49,169,218,28.7810,5.87,43,79,9778.19,deepseek
3,3,A bat and a ball cost $1.10 in total. The bat ...,**ĊLetĠmeĠdenoteĠtheĠcostĠofĠtheĠballĠasĠ\(ĠxĠ...,32,2048,2080,382.3409,5.36,51,79,9778.19,deepseek
4,4,If 5 machines take 5 minutes to make 5 widgets...,"Ġ100machines,eachmaking5widgetsper5minutes,how...",29,2048,2077,379.3808,5.40,51,80,9778.19,deepseek


In [10]:
llm_groq = ChatGroq(
    model="qwen/qwen3-32b",
    temperature=0.4,
    api_key=userdata.get("GROQ_API_KEY")
)

groq_records = []

for i in range(len(df)):
    start = time.time()
    output = llm_groq.invoke(df["prompt"][i])
    latency = time.time() - start

    # Groq gives token usage directly in metadata
    usage = output.response_metadata.get('token_usage', {})
    prompt_tokens    = usage.get('prompt_tokens', 0)
    output_tokens    = usage.get('completion_tokens', 0)
    total_tokens     = usage.get('total_tokens', 0)
    # use groq's own time if available, else use ours
    groq_latency     = usage.get('completion_time', latency)

    groq_records.append({
        "question_id"      : i,
        "question"         : df["prompt"][i],
        "answer"           : output.content,
        "prompt_tokens"    : prompt_tokens,
        "output_tokens"    : output_tokens,
        "total_tokens"     : total_tokens,
        "latency_sec"      : round(groq_latency, 4),
        "tokens_per_sec"   : round(output_tokens / groq_latency, 2) if groq_latency > 0 else 0,
        "gpu_usage_pct"    : None,   # cloud API — not applicable
        "gpu_temp_c"       : None,
        "gpu_mem_used_mb"  : None,
    })

    print(f"[{i+1}/{len(df)}] done | tokens: {total_tokens} | latency: {groq_latency:.2f}s")

groq_df = pd.DataFrame(groq_records)
groq_df['model'] = 'groq/qwen3-32b'
groq_df.to_csv('groq_metrics.csv', index=False)
print(groq_df.head())

[1/30] done | tokens: 1638 | latency: 3.83s
[2/30] done | tokens: 1503 | latency: 3.98s
[3/30] done | tokens: 2093 | latency: 4.23s
[4/30] done | tokens: 2090 | latency: 4.80s
[5/30] done | tokens: 1769 | latency: 3.23s
[6/30] done | tokens: 2120 | latency: 7.87s
[7/30] done | tokens: 1766 | latency: 2.97s
[8/30] done | tokens: 737 | latency: 1.72s
[9/30] done | tokens: 2089 | latency: 3.57s
[10/30] done | tokens: 2074 | latency: 5.25s
[11/30] done | tokens: 1130 | latency: 3.45s
[12/30] done | tokens: 685 | latency: 2.46s
[13/30] done | tokens: 1526 | latency: 3.38s
[14/30] done | tokens: 2092 | latency: 6.17s
[15/30] done | tokens: 2077 | latency: 5.64s
[16/30] done | tokens: 1112 | latency: 2.51s
[17/30] done | tokens: 1368 | latency: 4.27s
[18/30] done | tokens: 1721 | latency: 5.14s
[19/30] done | tokens: 537 | latency: 1.08s
[20/30] done | tokens: 2073 | latency: 4.24s
[21/30] done | tokens: 1165 | latency: 4.10s
[22/30] done | tokens: 274 | latency: 0.47s
[23/30] done | tokens: 

In [11]:
llm_mistral = ChatMistralAI(
    model="mistral-small-2506",
    temperature=0.4,
    max_tokens=2048,
    api_key=userdata.get("MISTRAL_API_KEY")
)

mistral_records = []

for i in range(len(df)):
    start = time.time()
    output = llm_mistral.invoke(df["prompt"][i])
    latency = time.time() - start

    # Debug on first row
    if i == 0:
        print("usage_metadata:", output.usage_metadata)

    # Same as Groq — usage_metadata is top level on output object
    prompt_tokens = output.usage_metadata.get('input_tokens', 0)
    output_tokens = output.usage_metadata.get('output_tokens', 0)
    total_tokens  = output.usage_metadata.get('total_tokens', 0)

    mistral_records.append({
        "question_id"     : i,
        "question"        : df["prompt"][i],
        "answer"          : output.content,
        "prompt_tokens"   : prompt_tokens,
        "output_tokens"   : output_tokens,
        "total_tokens"    : total_tokens,
        "latency_sec"     : round(latency, 4),
        "tokens_per_sec"  : round(output_tokens / latency, 2) if latency > 0 else 0,
        "gpu_usage_pct"   : None,
        "gpu_temp_c"      : None,
        "gpu_mem_used_mb" : None,
    })

    print(f"[{i+1}/{len(df)}] done | tokens: {total_tokens} | latency: {latency:.2f}s")

mistral_df = pd.DataFrame(mistral_records)
mistral_df['model'] = 'mistral-small-2506'
mistral_df.to_csv('mistral_metrics.csv', index=False)
print(mistral_df.head())

usage_metadata: {'input_tokens': 53, 'output_tokens': 1220, 'total_tokens': 1273}
[1/30] done | tokens: 1273 | latency: 8.27s
[2/30] done | tokens: 837 | latency: 6.41s
[3/30] done | tokens: 1058 | latency: 7.14s
[4/30] done | tokens: 1038 | latency: 7.47s
[5/30] done | tokens: 1084 | latency: 6.94s
[6/30] done | tokens: 1042 | latency: 6.75s
[7/30] done | tokens: 369 | latency: 2.79s
[8/30] done | tokens: 168 | latency: 1.30s
[9/30] done | tokens: 591 | latency: 3.95s
[10/30] done | tokens: 482 | latency: 3.57s
[11/30] done | tokens: 321 | latency: 2.13s
[12/30] done | tokens: 440 | latency: 3.13s
[13/30] done | tokens: 437 | latency: 2.92s
[14/30] done | tokens: 370 | latency: 3.85s
[15/30] done | tokens: 759 | latency: 5.51s
[16/30] done | tokens: 423 | latency: 2.95s
[17/30] done | tokens: 614 | latency: 3.87s
[18/30] done | tokens: 549 | latency: 3.57s
[19/30] done | tokens: 330 | latency: 2.02s
[20/30] done | tokens: 693 | latency: 4.60s
[21/30] done | tokens: 105 | latency: 0.83